# Create Research to Prevent Blindness Awards

Creates OpenAlex award rows from Research to Prevent Blindness's official public **Individual Grantees** archive.

**Awarding body:** Research to Prevent Blindness — `F4320306811` (US, ROR `https://ror.org/04drjs621`, DOI `10.13039/100001818`).

**Source:** `https://www.rpbusa.org/grantees/`, using the site's first-party WordPress AJAX endpoint for individual-grantee pagination.

**Source method:** method-5/3 hybrid: public server-rendered archive + first-party `admin-ajax.php?action=get_filtered_posts&post_type=individual-grantees`. The static `/page/N/` URLs duplicate the first page; the AJAX endpoint is the official pagination path used by the page JavaScript.

**Pattern:** individual research-award/grant pattern. RPB publishes named awardees, institution, award name, award year, amount, research area, and often a research description. The source does **not** publish native grant numbers, so `funder_award_id` is a deterministic source-derived key from visible fields; uniqueness is enforced in the scraper.

**Scope note:** this ingest covers the historical **Individual Grantees** archive only. The same page also shows Departmental Grantees, but that panel is labelled current-year-only; it is excluded to avoid mixing incomplete org-level rows into the historical individual-award corpus.

**Local validation on 2026-06-09:** 1,653 rows, years 1963-2025, 100% awardee/year/award/amount/currency, 84.0% institution, 53.2% research area, 50.8% description.

**S3:** `s3a://openalex-ingest/awards/rpb/rpb_grantees.parquet`

**Provenance:** `rpb_grantees` (production pre-flight count 0 on 2026-06-09).

**Priority:** 224 (next free Codex even slot after scanning open PRs through #258; odd slots 219/221/223/225 are held by SweCRIS PR #253).


## Step 1: Create raw table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rpb_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rpb/rpb_grantees.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS total_rpb_rows
FROM openalex.awards.rpb_raw;


## Step 1.5: Inspect raw data, money, dates, and native keys


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.rpb_raw;


In [ ]:
%sql
SELECT
    funder_award_id,
    raw_awardee_name,
    awardee_name_clean,
    given_name,
    family_name,
    institution_name,
    award_name,
    award_year,
    amount,
    currency,
    research_area,
    SUBSTRING(description, 1, 160) AS description_sample,
    landing_page_url
FROM openalex.awards.rpb_raw
LIMIT 10;


In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Do not wrap DESCRIBE in a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'rpb_raw'
  AND LOWER(column_name) RLIKE 'amount|amt|total|funding|award|grant|currency|usd|dollar|value|budget|cost'
ORDER BY ordinal_position;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(raw_awardee_name) AS has_awardee,
    COUNT(given_name) AS has_given,
    COUNT(family_name) AS has_family,
    COUNT(institution_name) AS has_institution,
    COUNT(award_name) AS has_award_name,
    COUNT(award_year) AS has_award_year,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(research_area) AS has_research_area,
    COUNT(description) AS has_description,
    ROUND(try_divide(COUNT(institution_name), COUNT(*)) * 100.0, 1) AS pct_institution,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.rpb_raw;


In [ ]:
%sql
SELECT
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year,
    COUNT(DISTINCT TRY_CAST(award_year AS INT)) AS distinct_award_years,
    COUNT(*) AS rows
FROM openalex.awards.rpb_raw;


In [ ]:
%sql
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    percentile_approx(TRY_CAST(amount AS DOUBLE), 0.5) AS median_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.rpb_raw;


In [ ]:
%sql
SELECT award_name, COUNT(*) AS rows,
       MIN(TRY_CAST(award_year AS INT)) AS first_year,
       MAX(TRY_CAST(award_year AS INT)) AS last_year,
       ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_usd
FROM openalex.awards.rpb_raw
GROUP BY award_name
ORDER BY rows DESC, award_name;


In [ ]:
%sql
-- §6.4a-style repeated-recipient check: repeated names can be legitimate repeat awardees,
-- but high-frequency names should be eyeballed for parsing mistakes.
SELECT given_name, family_name, COUNT(*) AS awards,
       COUNT(DISTINCT award_name) AS distinct_award_names,
       MIN(TRY_CAST(award_year AS INT)) AS first_year,
       MAX(TRY_CAST(award_year AS INT)) AS last_year
FROM openalex.awards.rpb_raw
WHERE family_name IS NOT NULL
GROUP BY given_name, family_name
ORDER BY awards DESC, family_name, given_name
LIMIT 25;


## Step 1.6: Fail-fast funder existence assertion


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320306811;


In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for Research to Prevent Blindness (F4320306811)'
) AS rpb_funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320306811;


## Step 2: Transform to OpenAlex awards schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rpb_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306811
), cleaned AS (
    SELECT
        *,
        TRY_CAST(award_year AS INT) AS award_year_int,
        TRY_CAST(amount AS DOUBLE) AS amount_double
    FROM openalex.awards.rpb_raw
    WHERE funder_award_id IS NOT NULL
), awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':', LOWER(TRIM(c.funder_award_id))
        ))) % 9000000000 AS id,
        CONCAT('RPB ', c.award_name, ' - ', c.awardee_name_clean,
               CASE WHEN c.award_year_int IS NOT NULL THEN CONCAT(' (', c.award_year_int, ')') ELSE '' END) AS display_name,
        c.description AS description,
        f.funder_id,
        c.funder_award_id,
        CASE WHEN c.amount_double > 0 THEN c.amount_double ELSE NULL END AS amount,
        CASE WHEN c.amount_double > 0 THEN 'USD' ELSE NULL END AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'grant' AS funding_type,
        c.award_name AS funder_scheme,
        'rpb_grantees' AS provenance,
        CAST(NULL AS DATE) AS start_date,
        CAST(NULL AS DATE) AS end_date,
        CASE
            WHEN c.award_year_int > YEAR(current_date()) + 1 THEN NULL
            ELSE c.award_year_int
        END AS start_year,
        CAST(NULL AS INT) AS end_year,
        CASE
            WHEN c.family_name IS NOT NULL OR c.given_name IS NOT NULL OR c.institution_name IS NOT NULL THEN struct(
                c.given_name AS given_name,
                c.family_name AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    c.institution_name AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
            ELSE NULL
        END AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        c.landing_page_url,
        CAST(NULL AS STRING) AS doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               TRY_CAST(abs(xxhash64(CONCAT(
                   TRY_CAST(f.funder_id AS STRING), ':', LOWER(TRIM(c.funder_award_id))
               ))) % 9000000000 AS STRING)) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM cleaned c
    CROSS JOIN funder_resolved f
)
SELECT *
FROM awards_transformed;


## Step 3: Delete/insert into openalex_awards_raw


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rpb_grantees' AND priority = 224;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    224 AS priority
FROM openalex.awards.rpb_awards;


## Step 6: Verification

§6.7 amount check applies and should pass: the RPB archive publishes a positive USD amount for every individual award row in local validation. Institution coverage is partial because early/current cards do not always publish an institution; country is intentionally NULL because the source does not publish a recipient-country field.


In [ ]:
%sql
SELECT COUNT(*) AS total_rows
FROM openalex.awards.rpb_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.rpb_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(lead_investigator) AS has_lead,
    COUNT(lead_investigator.family_name) AS has_family,
    COUNT(lead_investigator.affiliation.name) AS has_affiliation,
    COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) AS pct_affiliation
FROM openalex.awards.rpb_awards;


In [ ]:
%sql
SELECT
    MIN(start_year) AS first_year,
    MAX(start_year) AS last_year,
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.rpb_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(SUM(amount), 0) AS total_usd,
    collect_set(currency) AS currencies
FROM openalex.awards.rpb_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS rows,
       MIN(start_year) AS first_year, MAX(start_year) AS last_year,
       ROUND(SUM(amount), 0) AS total_usd,
       ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.rpb_awards
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme;


In [ ]:
%sql
-- Confirm no records slipped past the future-year cap.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.rpb_awards
WHERE start_year > YEAR(current_date()) + 1
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
-- §6.4a repeated-recipient QA. A few RPB awardees appear more than five times locally; verify repeated names are real repeat awardees.
SELECT
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    COUNT(*) AS awards,
    COUNT(DISTINCT funder_scheme) AS distinct_schemes,
    MIN(start_year) AS first_year,
    MAX(start_year) AS last_year,
    collect_set(funder_scheme) AS schemes
FROM openalex.awards.rpb_awards
WHERE lead_investigator.family_name IS NOT NULL
GROUP BY lead_investigator.given_name, lead_investigator.family_name
ORDER BY awards DESC, family_name, given_name
LIMIT 25;


In [ ]:
%sql
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.rpb_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
SELECT
    id,
    SUBSTRING(display_name, 1, 90) AS title,
    funder_scheme,
    start_year,
    amount,
    currency,
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    lead_investigator.affiliation.name AS institution
FROM openalex.awards.rpb_awards
ORDER BY start_year DESC, funder_scheme, family_name
LIMIT 20;


## Contractor handoff / admin notes

Contractor-local validation completed with `scripts/local/rpb_to_s3.py --skip-upload`. Contractor has no S3/Databricks access, so admin must upload the parquet, run this notebook, inspect the Step 6 cells, then decide whether to wire a scheduled job after QA.
